### Regional / Global Final Preparation
**Task:** Sequence Labeling (Named Entity Recognition) with Imbalanced Data Mitigation
**Mini-Research:** Low-Rank Adaptation (LoRA) integration based on *Hu et al.*

>*Complete all codes, read the paper hints, and finish training without using legacy libraries like `torchtext`.*

##### Task 1: Environment & Dataset
We will use the generic `CoNLL-2003` dataset sourced from Hugging Face for Sequence Labeling. This mimics large-scale offline JSONl extraction commonly found in global finals.

In [ ]:
# ======================= 1-1: Environment Setup ==================
%pip install -q transformers datasets peft seqeval

In [ ]:
from datasets import load_dataset

# ======================= 1-2: Download Dataset ==================
dataset = load_dataset("conll2003")
print("Dataset Terunduh:\n", dataset)
# ===============================================================

##### Task 2: Addressing Imbalanced Data (Challenge)
In NER, the `O` (Outside) tag often makes up >80% of the corpus. The model might achieve 90% "accuracy" just by guessing `O`. We need to compute "Class Weights" dynamically to penalize the model heavily if it gets minor entities (Person, Org) wrong.

In [ ]:
import torch
import numpy as np
from collections import Counter

ner_tags_list = dataset['train']['ner_tags']
all_tags = [tag for tags in ner_tags_list for tag in tags]
tag_counts = Counter(all_tags)

total_tags = sum(tag_counts.values())
num_classes = len(tag_counts)

# ======================= 2-1: Calculate Class Weights ============
# Rumus invers frekuensi: Weight_Class = Total_Data / (Num_Classes * Count_Class)
class_weights = []
for i in range(num_classes):
    weight = ___________  # COMPLETE THIS CODE (Gunakan formula di atas)
    class_weights.append(weight)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Bobot Kelas (Class Weights):\n", class_weights_tensor)
# =================================================================

##### Task 3: Preprocessing & Token Alignment (WordPiece Challenge)
Replacing `torchtext` with modern tokenizer map. BERT tokenization slices words (e.g. "Huawei" -> "Hua", "##wei"). You must map label IDs so that subwords are ignored (-100) and only the first root token receives the original class.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# ======================= 3-1: Tokenizer Alignment ==================
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Special token ([CLS], [SEP]) -> Disembunyikan dari loss
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Root word token -> Berikan label asli
                label_ids.append(___________) # COMPLETE THIS
            else:
                # Sub-word token (e.g. ##wei) -> Disembunyikan
                label_ids.append(___________) # COMPLETE THIS
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
# =================================================================

##### Task 4: Custom PyTorch DataLoaders
Implement pure PyTorch DataLoader combined with HF `DataCollatorForTokenClassification` for out-of-the-box dynamic padding, making `torchtext` obsolete.

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForTokenClassification

# Convert formats to PyTorch tensors
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# ======================= 4-1: DataLoader Integration ==================
data_collator = DataCollatorForTokenClassification(tokenizer)

train_dataloader = DataLoader(
    ___________, # COMPLETE THIS (Pass training split)
    shuffle=True, 
    collate_fn=data_collator, 
    batch_size=16
)
valid_dataloader = DataLoader(
    ___________, # COMPLETE THIS (Pass validation split)
    collate_fn=data_collator, 
    batch_size=16
)
# ====================================================================

##### Task 5: Mini-Research on PEFT (LoRA)
According to the paper *LoRA: Low-Rank Adaptation of Large Language Models (Hu et al., 2021)*, inserting rank-decomposition matrices into Transformer layers achieves comparable performance to full finetuning while cutting trainable parameters by ~99%.

Instead of unfreezing all BERT layers, configure LoRA explicitly for a Sequence Labeling (Token Classification) task.

In [ ]:
from transformers import AutoModelForTokenClassification
from peft import get_peft_model, LoraConfig, TaskType

# ======================= 5-1: LoRA Init ==================
model = AutoModelForTokenClassification.from_pretrained("bert-base-uncased", num_labels=num_classes)

peft_config = LoraConfig(
    task_type=TaskType.___________,      # COMPLETE THIS (e.g. TOKEN_CLS)
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=[___________]         # COMPLETE THIS (Which BERT attention modules are targeted? "query", "value", etc)
)

lora_model = get_peft_model(model, peft_config)
lora_model.print_trainable_parameters()
# ============================================================

##### Task 6: Custom Loss Super-Loop (Mitigating Imbalance)
Because we use PEFT and custom data ratios, you will override the model's auto-loss by explicitly calculating it via `nn.CrossEntropyLoss(weight=...)`.

In [ ]:
import torch.optim as optim
import torch.nn as nn
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lora_model.to(device)
class_weights_tensor = class_weights_tensor.to(device)

optimizer = optim.AdamW(lora_model.parameters(), lr=1e-3) # LoRA usually needs higher LR

# ======================= 6-1: Unbalanced Class Loss ==================
loss_fct = nn.CrossEntropyLoss(weight=___________) # COMPLETE THIS
# ===================================================================

EPOCHS = 3

for epoch in range(EPOCHS):
    lora_model.train()
    total_loss = 0
    batch_loader = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")
    
    for batch in batch_loader:
        inputs = batch['input_ids'].to(device)
        masks = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = lora_model(inputs, attention_mask=masks)
        logits = outputs.logits
        
        # Meratakan matrix untuk menghitung Loss manual
        # Agar padding label (-100) bisa terfilter
        active_loss = masks.view(-1) == 1
        active_logits = logits.view(-1, num_classes)[active_loss]
        active_labels = labels.view(-1)[active_loss]
        
        loss = loss_fct(active_logits, active_labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        batch_loader.set_postfix({'Step Loss': f"{loss.item():.4f}"})
        
    print(f"Avg Epoch {epoch+1} Loss: {total_loss/len(train_dataloader):.4f}")